# Análise de Indicadores de Saúde Pública no Brasil (2015–2024)
**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Aluno:** Yago Baltazar Ferreira  
**Professor:** Alexandre Neves Louzada

## 1. Introdução
Este notebook investiga indicadores de saúde pública no Brasil entre 2015 e 2024 a partir de uma base **simulada** fornecida pelo professor. O objetivo é analisar a evolução dos indicadores, comparar regiões e estados, avaliar a capacidade hospitalar e identificar as áreas mais vulneráveis.

**Perguntas orientadoras**
- Quais regiões apresentam maiores índices de mortalidade?
- Houve melhoria nos indicadores ao longo do tempo?
- Existem desigualdades regionais?
- Quais estados possuem maior taxa de internação?
- Existe relação entre vacinação e redução de doenças?
- Como evoluiu a expectativa de vida?
- Quais indicadores apresentam maior criticidade?

## 2. Contextualização da saúde pública
Indicadores de saúde pública avaliam a qualidade de vida da população e apoiam decisões de governo. Mortalidade e expectativa de vida mostram o resultado final da saúde, vacinação mostra a prevenção e médicos e leitos mostram a infraestrutura disponível.

## 3. Explicação da base
Arquivo: `simulacao_saude_publica_brasil.csv` (4.440 linhas, 14 colunas), com 37 municípios de 20 estados, de janeiro de 2015 a dezembro de 2024.

| Coluna | Descrição |
|---|---|
| ano, mes, data | Período da medição |
| regiao, uf, municipio | Localização |
| expectativa_vida | Expectativa de vida (anos) |
| taxa_mortalidade | Taxa de mortalidade |
| taxa_internacao | Taxa de internação |
| cobertura_vacinal | Percentual de vacinação |
| medicos_por_1000 | Médicos por 1000 habitantes |
| leitos_hospitalares | Quantidade de leitos |
| casos_doencas_cronicas | Quantidade estimada de casos |
| nivel_criticidade | Baixo, Médio, Alto ou Crítico |

## 4. Leitura dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("../dados/simulacao_saude_publica_brasil.csv", encoding="utf-8-sig")
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe().round(2)

## 5. Limpeza e preparação

In [ ]:
print("Valores ausentes:", df.isna().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())
print("Regiões:", sorted(df["regiao"].unique()))
print("Níveis de criticidade:", df["nivel_criticidade"].unique())

df["data"] = pd.to_datetime(df["data"])
ordem = ["Baixo", "Médio", "Alto", "Crítico"]
df["nivel_criticidade"] = pd.Categorical(df["nivel_criticidade"], categories=ordem, ordered=True)

colunas_indicadores = ["expectativa_vida", "taxa_mortalidade", "taxa_internacao", "cobertura_vacinal",
                       "medicos_por_1000", "leitos_hospitalares", "casos_doencas_cronicas"]
df[colunas_indicadores].describe().loc[["min", "max"]]

A base está limpa: não há valores ausentes nem duplicados, e os valores estão em faixas plausíveis (por exemplo, cobertura vacinal abaixo de 100%). As principais etapas foram converter a coluna `data` para data e ordenar o nível de criticidade.

## 6. Engenharia de atributos

In [ ]:
df["criticidade_num"] = df["nivel_criticidade"].map({"Baixo": 1, "Médio": 2, "Alto": 3, "Crítico": 4}).astype(int)
df["periodo"] = np.where(df["ano"] <= 2019, "2015-2019", "2020-2024")

def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

por_uf = df.groupby(["uf", "regiao"])[colunas_indicadores].mean().reset_index()
por_uf["indice_vulnerabilidade"] = (
    normalizar(por_uf["taxa_mortalidade"]) + normalizar(por_uf["taxa_internacao"])
    + (1 - normalizar(por_uf["cobertura_vacinal"])) + (1 - normalizar(por_uf["expectativa_vida"]))
    + (1 - normalizar(por_uf["medicos_por_1000"]))
) / 5
ranking = por_uf.sort_values("indice_vulnerabilidade", ascending=False)
ranking[["uf", "regiao", "indice_vulnerabilidade"]].round(3).head(10)

O **índice de vulnerabilidade** (0 a 1) combina cinco indicadores já normalizados: mortalidade e internação altas, e vacinação, expectativa de vida e médicos baixos. Quanto maior, mais vulnerável o estado.

## 7. KPIs

In [ ]:
print(f"Expectativa média de vida: {df['expectativa_vida'].mean():.1f} anos")
print(f"Taxa média de mortalidade: {df['taxa_mortalidade'].mean():.2f}")
print(f"Cobertura vacinal média: {df['cobertura_vacinal'].mean():.1f}%")
print(f"Estado mais vulnerável: {ranking.iloc[0]['uf']} ({ranking.iloc[0]['regiao']})")
print(f"Média de leitos hospitalares: {df['leitos_hospitalares'].mean():,.0f}")
print(f"Total de internações (soma das taxas): {df['taxa_internacao'].sum():,.0f}")

## 8. Visualizações

### 8.1 Evolução temporal

In [ ]:
por_ano = df.groupby(["ano", "regiao"])["taxa_mortalidade"].mean().reset_index()
plt.figure(figsize=(10, 4))
sns.lineplot(data=por_ano, x="ano", y="taxa_mortalidade", hue="regiao", marker="o")
plt.title("Taxa média de mortalidade por região")
plt.show()

In [ ]:
vida_ano = df.groupby("ano")["expectativa_vida"].mean()
plt.figure(figsize=(10, 3.5))
sns.lineplot(x=vida_ano.index, y=vida_ano.values, marker="o")
plt.title("Expectativa média de vida por ano")
plt.ylabel("Anos")
plt.show()
vida_ano.round(2)

### 8.2 Comparação entre regiões e estados

In [ ]:
mort_uf = df.groupby(["uf", "regiao"])["taxa_mortalidade"].mean().reset_index().sort_values("taxa_mortalidade", ascending=False)
plt.figure(figsize=(10, 4))
sns.barplot(data=mort_uf, x="uf", y="taxa_mortalidade", hue="regiao", dodge=False)
plt.title("Mortalidade média por estado")
plt.show()

In [ ]:
int_uf = df.groupby("uf")["taxa_internacao"].mean().sort_values(ascending=False).reset_index()
plt.figure(figsize=(10, 4))
sns.barplot(data=int_uf, x="uf", y="taxa_internacao", hue="uf", legend=False)
plt.title("Taxa média de internação por estado")
plt.show()

### 8.3 Infraestrutura hospitalar

In [ ]:
infra = df.groupby("uf")[["leitos_hospitalares", "medicos_por_1000"]].mean().reset_index()
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=infra.sort_values("leitos_hospitalares", ascending=False), y="uf", x="leitos_hospitalares", hue="uf", legend=False, ax=eixos[0])
sns.barplot(data=infra.sort_values("medicos_por_1000", ascending=False), y="uf", x="medicos_por_1000", hue="uf", legend=False, ax=eixos[1])
eixos[0].set_title("Leitos hospitalares (média)")
eixos[1].set_title("Médicos por 1000 habitantes (média)")
plt.tight_layout()
plt.show()

### 8.4 Análise epidemiológica

In [ ]:
tabela = df.pivot_table(index="uf", columns="ano", values="taxa_mortalidade", aggfunc="mean")
plt.figure(figsize=(11, 6))
sns.heatmap(tabela, annot=True, fmt=".1f", cmap="YlOrRd", annot_kws={"size": 7})
plt.title("Heatmap epidemiológico: mortalidade por estado e ano")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.scatterplot(data=df, x="cobertura_vacinal", y="taxa_mortalidade", hue="regiao", alpha=0.5, s=18)
sns.regplot(data=df, x="cobertura_vacinal", y="taxa_mortalidade", scatter=False, color="black")
plt.title("Vacinação x mortalidade")
plt.show()
print("Correlação:", round(df["cobertura_vacinal"].corr(df["taxa_mortalidade"]), 3))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[colunas_indicadores].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação entre indicadores")
plt.show()

### 8.5 Criticidade

In [ ]:
prop = df.groupby(["regiao", "nivel_criticidade"], observed=True).size().unstack(fill_value=0)
prop = prop.div(prop.sum(axis=1), axis=0) * 100
prop.plot(kind="bar", stacked=True, colormap="RdYlGn_r", figsize=(9, 4))
plt.ylabel("% dos registros")
plt.title("Distribuição da criticidade por região")
plt.show()
prop.round(1)

### 8.6 Tabela dinâmica

In [ ]:
df.pivot_table(index="regiao", columns="nivel_criticidade", values="taxa_mortalidade", aggfunc="mean", observed=True).round(2)

## 9. Interpretação dos resultados
- **Mortalidade por região:** as médias regionais são muito próximas (entre 7,9 e 8,1), então não há desigualdade regional marcante. Por estado, a maior média é de MS e a menor é de AM.
- **Evolução temporal:** a expectativa média de vida oscila entre 73,2 e 73,8 anos sem tendência clara de melhora ou piora. Os demais indicadores também variam pouco de um ano para outro.
- **Internação:** o DF tem a maior taxa média de internação e TO a menor.
- **Vacinação x mortalidade:** a correlação é praticamente zero (cerca de 0,02). Nesta base simulada, não há evidência de que mais vacinação reduza a mortalidade. Isso difere do que se espera na realidade, e acontece porque os dados são simulados.
- **Infraestrutura:** leitos e médicos variam entre estados, mas não se relacionam com internação nem com mortalidade.
- **Criticidade:** cerca de 24% dos registros são Críticos, distribuídos de forma semelhante entre as regiões.
- **Estado mais vulnerável:** pelo índice, MS aparece no topo, mas o índice compara estados que são parecidos entre si, então a diferença deve ser lida com cautela.

## 10. Conclusão
A base simulada mostra indicadores bastante homogêneos entre regiões e ao longo dos anos, sem relações fortes entre as variáveis. O principal aprendizado é metodológico: construir KPIs, um índice de vulnerabilidade e visualizações comparativas permite identificar onde olhar primeiro (estados no topo do ranking de vulnerabilidade) e também reconhecer quando os dados **não** sustentam uma conclusão, como a relação entre vacinação e mortalidade.

Como próximos passos, o ideal seria repetir a análise com dados reais (por exemplo, DATASUS e IBGE) e incluir populações para calcular taxas absolutas de internação.

*Observação: base simulada, apenas para fins educacionais.*